# 🩺 Health Claims Fact-Checker: DistilBERT Fine-Tuning Notebook

This notebook will fine-tune a pre-trained **DistilBERT** model from Hugging Face on the health claims fact-checking dataset. It will achieve significantly higher accuracy than traditional TF-IDF + Machine Learning approaches by capturing complex sentence context.

## 🚀 Instructions for Running in Google Colab:
1. Open [Google Colab](https://colab.research.google.com/).
2. Upload this notebook file (`train_colab.ipynb`).
3. Set the runtime to use a GPU:
   * Go to **Runtime** > **Change runtime type**.
   * Select **T4 GPU** (or any other available GPU) under Hardware accelerator.
4. Upload the `train.tsv` file from your project folder using the folder icon on the left panel.
5. Run all cells sequentially. Once completed, a zip file named `distilbert_health_model.zip` will be generated. Download it, extract it to a folder named `distilbert_health_model` in your project workspace directory, and run the Streamlit app!

In [ ]:
# 1. Install required packages
!pip install transformers accelerate scikit-learn pandas tqdm evaluate

In [ ]:
# 2. Import libraries and check for GPU
import os
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import (
    DistilBertTokenizerFast, 
    DistilBertForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    EarlyStoppingCallback
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("\n⚠️ WARNING: GPU not detected. Please select T4 GPU in 'Runtime -> Change runtime type' to speed up training.")

In [ ]:
# 3. Load and clean the dataset
if not os.path.exists("train.tsv"):
    raise FileNotFoundError("Please upload 'train.tsv' using the file icon on the left panel before running this cell.")

print("Loading dataset...")
df = pd.read_csv('train.tsv', sep='\t').dropna(subset=['claim', 'label']).rename(columns={'claim': 'text'})
valid_labels = ['true', 'false', 'mixture', 'unproven']
train_df = df[df['label'].isin(valid_labels)].copy()
print(f"Dataset loaded. Total clean samples: {len(train_df)}")

# Map string labels to numeric classes
label_map = {'true': 0, 'false': 1, 'mixture': 2, 'unproven': 3}
train_df['label_idx'] = train_df['label'].map(label_map)
print("Class distribution:")
print(train_df['label'].value_counts())

In [ ]:
# 4. Split the dataset (70% train / 30% test)
X_train_texts, X_test_texts, y_train_labels, y_test_labels = train_test_split(
    train_df['text'].values,
    train_df['label_idx'].values,
    test_size=0.3,
    random_state=42,
    stratify=train_df['label_idx'].values
)
print(f"Training split: {len(X_train_texts)} samples")
print(f"Testing split: {len(X_test_texts)} samples")

In [ ]:
# 5. Load DistilBERT Tokenizer and tokenize the datasets
print("Loading DistilBERT tokenizer...")
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

print("Tokenizing text inputs (max_length=128)...")
train_encodings = tokenizer(list(X_train_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(X_test_texts), truncation=True, padding=True, max_length=128)

In [ ]:
# 6. Define PyTorch Dataset wrapper
class HealthClaimsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HealthClaimsDataset(train_encodings, y_train_labels)
test_dataset = HealthClaimsDataset(test_encodings, y_test_labels)

In [ ]:
# 7. Load model and define metrics evaluator
print("Initializing DistilBERT model for sequence classification...")
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=4)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
# 8. Define Hugging Face TrainingArguments and Trainer
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
# 9. Fine-tune the model
print("Starting fine-tuning process...")
trainer.train()

In [ ]:
# 10. Generate and print final evaluation metrics
print("Evaluating fine-tuned model...")
eval_results = trainer.predict(test_dataset)
predictions = np.argmax(eval_results.predictions, axis=1)

print("\n=== FINAL EVALUATION METRICS ===")
print(classification_report(y_test_labels, predictions, target_names=valid_labels))
print("Confusion Matrix:")
print(confusion_matrix(y_test_labels, predictions))

In [ ]:
# 11. Save model, tokenizer, and zip them
output_dir = './distilbert_health_model'
print(f"Saving trained model to {output_dir}...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

import shutil
shutil.make_archive('distilbert_health_model', 'zip', output_dir)
print("\n✅ SUCCESS!")
print("The file 'distilbert_health_model.zip' is ready for download in the left file panel.")
print("Download and extract it to your local project directory as a folder named 'distilbert_health_model'.")